Training pipeline:
Preparation for training + Model + Training

Inference pipeline:
Model + Inference

# I. Data Preparation

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

In [ ]:
!pip install huggingface_hub

## Loading dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("VLyb/YAGO3-10")

To work with the dataset triples use:

```
for triple in train_ds:
    # triple = {'head': ..., 'relation': ..., 'tail': ...}
    h, r, t = triple.values()
```

## Working with Data

## Visualization of Samples from Dataset

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

In [ ]:
# Read splits
train_ds = ds['train']
val_ds = ds['validation']
test_ds = ds['test']

In [ ]:
edge_list = []

for triple in test_ds:
    head, edge, tail = triple.values()
    edge_list.append([head, tail])

In [ ]:
G = nx.Graph(edge_list)

In [ ]:
from collections import deque
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np


def sample_neighborhood(G: nx.Graph, n: int = 15):
    nodes = list(G.nodes())
    start = np.random.choice(nodes)
    sampled_nodes = [start]
    q = deque([start])

    while len(q) != 0:
        u = q.popleft()
        neighborhood = list(nx.neighbors(G, u))

        if len(sampled_nodes) + len(neighborhood) >= n:
            diff = n - len(sampled_nodes)
            sampled_nodes.extend(neighborhood[:diff])
            return sampled_nodes, start  # Return both nodes and start node

        for v in neighborhood:
            if v not in sampled_nodes:  # Avoid duplicates
                sampled_nodes.append(v)
                q.append(v)

    return sampled_nodes, start

# Sample multiple components
node_samples = []
start_nodes = []

for _ in range(15):
    nodes, start = sample_neighborhood(G, n=32)
    node_samples.extend(nodes)
    start_nodes.append(start)

G_n = nx.subgraph(G, node_samples)

# Visualization with labels only on start nodes
options = {
    'node_color': 'blue',
    'node_size': 30,
    'width': 1,
    'arrowstyle': '-|>',
    'arrowsize': 10
}

plt.figure(figsize=(10, 8))

pos = nx.spring_layout(G_n)
nx.draw_networkx(G_n, pos, arrows=True, **options, with_labels=False)

# Label only the start nodes of each component
labels = {node: node for i, node in enumerate(start_nodes) if node in G_n}
nx.draw_networkx_labels(G_n, pos, labels=labels, font_size=8, font_weight='bold',
                       font_color='green')  # Make them stand out

plt.title("Knowledge Subgraph of YAGO3-10")
plt.axis('off')
plt.show()

## Dataset statistics

Let's see how many connected components we can see

In [ ]:
len(list(nx.connected_components(G)))

## Preparing train/evaluation triples

In [ ]:
import torch
from typing import Iterable, Dict, Tuple, Any, List, FrozenSet

### Convertion from node/edge name to an index

For faster access and indexing, convert node and edge labels to corresponding indices across the entire KG.

In [ ]:
def build_node_indexer(triples: Iterable[Dict[str, str]]) -> Tuple[Dict[str, int], Dict[int, str]]:

    nodes = set()
    for triple in triples:
        h, r, t = triple.values()
        nodes.add(h)
        nodes.add(t)

    nodes = list(nodes)
    node2idx = {node: i for i, node in enumerate(nodes)}
    idx2node = {i: node for node, i in node2idx.items()}

    return node2idx, idx2node

In [ ]:
def build_edge_indexer(triples: Iterable[Dict[str, str]]) -> Tuple[Dict[str, int], Dict[int, str]]:

    edges = set()
    for triple in triples:
        h, r, t = triple.values()
        edges.add(r)

    edges = list(edges)
    edge2idx = {edge: i for i, edge in enumerate(edges)}
    idx2edge = {i: edge for edge, i in edge2idx.items()}

    return edge2idx, idx2edge

In [ ]:
total_ds = train_ds.to_list() + val_ds.to_list() + test_ds.to_list()

In [ ]:
node2idx, idx2node = build_node_indexer(total_ds)

In [ ]:
edge2idx, idx2edge = build_edge_indexer(total_ds)

In [ ]:
def apply_index(triples: Iterable[Dict[str, str]], node2idx: Dict[str, int], edge2idx: Dict[str, int]) -> Iterable[Dict[str, int]]:
    # Convert triples to numerical indices
    numerical_triples = []
    for triple in triples:
        h, r, t = triple.values()
        numerical_triples.append({
            'head': node2idx[h],
            'relation': edge2idx[r],
            'tail': node2idx[t]
        })
    return numerical_triples

We will be using the following data for further generation of datasets(may take some time).

In [ ]:
numerical_train_ds = apply_index(train_ds, node2idx, edge2idx)
numerical_val_ds = apply_index(val_ds, node2idx, edge2idx)
numerical_test_ds = apply_index(test_ds, node2idx, edge2idx)

Dump indices into files for future reuse and consistency of generation:

In [ ]:
import json

with open('node2idx.json', 'w') as f:
    json.dump(node2idx, f)

with open('idx2node.json', 'w') as f:
    json.dump(idx2node, f)

with open('edge2idx.json', 'w') as f:
    json.dump(edge2idx, f)

with open('idx2edge.json', 'w') as f:
    json.dump(idx2edge, f)

## Training Stage $-$ Negative Sampling

Generate invalid triples by swapping either a head or a tail for copy of each existing triple in the train set with any entity present in the whole dataset.

As a result, we obtain both true and corrupted triples. For each triple, we assign a score ($1$ for valid, $0$ - for corrupted). The function returns the new training triples and the corresponding scores.

In [ ]:
import tqdm

def create_training_dataset(
    train_triples: Iterable[Dict[str, int]],
    all_entities: List[int],
    negative_ratio=1
) -> Tuple[List[Dict[str, int]], List[int]]:

    # Step 1: Positive examples (from original train split)
    positives = train_triples
    pos_labels = [1] * len(positives)

    # Step 2: Generate negative examples (NO FILTERING)
    negatives = []

    progress_bar = tqdm.tqdm(total=len(positives), leave=True, position=0)

    for triple in positives:
        h, r, t = triple.values()
        for i in range(negative_ratio):
            if np.random.random() > 0.5:
                # Corrupt head
                h_neg = np.random.choice(all_entities)
                while h_neg == h:
                    h_neg = np.random.choice(all_entities)
                negatives.append({
                    'head': h_neg,
                    'relation': r,
                    'tail': t
                })
            else:
                # Corrupt tail
                t_neg = np.random.choice(all_entities)
                while t_neg == t:
                    t_neg = np.random.choice(all_entities)
                negatives.append({
                    'head': h,
                    'relation': r,
                    'tail': t_neg
                })
        progress_bar.update(1)

    neg_labels = [0] * len(negatives)

    # Step 3: Combine and shuffle
    all_triples = positives + negatives
    all_labels = pos_labels + neg_labels

    # Shuffle
    indices = list(range(len(all_triples)))
    np.random.shuffle(indices)
    all_triples = [all_triples[i] for i in indices]
    all_labels = [all_labels[i] for i in indices]

    return all_triples, all_labels

May take some time

In [ ]:
train_triples, train_labels = create_training_dataset(
    numerical_train_ds,
    list(idx2node.keys()),  # From train+val+test
    negative_ratio=1  # 1 negative per positive
)

Dump train triples and labels into json files:

In [ ]:
serializable_train_triples = []

for triple in train_triples:
    serializable_triple = {}
    serializable_triple['head'] = int(triple['head'])
    serializable_triple['relation'] = int(triple['relation'])
    serializable_triple['tail'] = int(triple['tail'])

    serializable_train_triples.append(serializable_triple)

In [ ]:
serializable_train_labels = [int(train_label) for train_label in train_labels]

In [ ]:
import json

with open('train_triples.json', 'w') as f:
    json.dump(serializable_train_triples, f)

with open('train_labels.json', 'w') as f:
    json.dump(serializable_train_labels, f)

## Evaluation Stage $-$ Filtration

The idea behind the evaluation stage is slightly different: for each test triple, we want to provide all possible (but not already present in the KG) candidates by swapping the head or the tail with any entity in the KG. In reality, modelling all terms is computationally expensive, so that the batch is limited to some value, e.g. $100$ corrupted candidates.

The resulting model will take these $100$ negative and $1$ positive triples and assign each relevance score.

Since we need to evaluate model performance on head and tail prediction separately, we create a separate lists of corrupted candidates.

In [ ]:
import tqdm

def create_evaluation_dataset(
    eval_triples: Iterable[Dict[str, int]],
    all_true_triples: FrozenSet[Tuple[int, int, int]],
    all_entities: List[int],
    batch_size: int = 100
):
    evaluation_batches = []

    progress_bar = tqdm.tqdm(total=len(eval_triples), leave=True, position=0)

    for test_triple in eval_triples:
        h, r, t = test_triple.values()

        # Create candidate sets
        tail_candidates = []
        head_candidates = []

        # Tail prediction: (h, r, ?)
        for entity in all_entities:
            if entity != t and (h, r, entity) not in all_true_triples:
                tail_candidates.append((h, r, entity))
            progress_bar.set_postfix(tail_count=f"{len(tail_candidates)}/{batch_size}")
            if len(tail_candidates) >= batch_size:
                break

        # Head prediction: (?, r, t)
        for entity in all_entities:
            if entity != h and (entity, r, t) not in all_true_triples:
                head_candidates.append((entity, r, t))
            progress_bar.set_postfix(head_count=f"{len(head_candidates)}/{batch_size}")
            if len(head_candidates) >= batch_size:
                break

        progress_bar.update(1)

        evaluation_batches.append({
            'test_triple': test_triple,
            'tail_candidates': tail_candidates,
            'head_candidates': head_candidates
        })

    return evaluation_batches

In [ ]:
all_true_triples = set((
    node2idx[triple['head']],
    edge2idx[triple['relation']],
    node2idx[triple['tail']]
) for triple in total_ds)

Construct validation batches (may take some time):

In [ ]:
validation_batches = create_evaluation_dataset(
    numerical_val_ds,
    all_true_triples,
    list(idx2node.keys())
)

Dump validation batches into json files:

In [ ]:
import json

with open('validation_batches.json', 'w') as f:
    json.dump(validation_batches, f)

Construct test batches (may take some time):

In [ ]:
test_batches = create_evaluation_dataset(
    numerical_test_ds,
    all_true_triples,
    list(idx2node.keys())
)

Dump test batches into json files:

In [ ]:
import json

with open('test_batches.json', 'w') as f:
    json.dump(test_batches, f)

## Loading Pre-computed Splits

Here you can see the example of files downloading

In [ ]:
!git clone https://github.com/DavidVista/WGE4YAGO10

In [ ]:
import shutil

shutil.unpack_archive("WGE4YAGO10/dataset.zip", "dataset")

Load index

In [ ]:
import json

with open('dataset/dkr_project/node2idx.json', 'r') as f:
    node2idx = json.load(f)

with open('dataset/dkr_project/idx2node.json', 'r') as f:
    idx2node = json.load(f)

with open('dataset/dkr_project/edge2idx.json', 'r') as f:
    edge2idx = json.load(f)

with open('dataset/dkr_project/idx2edge.json', 'r') as f:
    idx2edge = json.load(f)

Load train triples and labels

In [ ]:
import json

with open('dataset/dkr_project/train_triples.json', 'r') as f:
    train_triples = json.load(f)

with open('dataset/dkr_project/train_labels.json', 'r') as f:
    train_labels = json.load(f)

Load test and validation batches

In [ ]:
import json

with open('dataset/dkr_project/validation_batches.json', 'r') as f:
    validation_batches = json.load(f)

In [ ]:
import json

with open('dataset/dkr_project/test_batches.json', 'r') as f:
    test_batches = json.load(f)

Create numeric datasets (may take some time)

In [ ]:
numerical_train_ds = apply_index(train_ds, node2idx, edge2idx)
numerical_val_ds = apply_index(val_ds, node2idx, edge2idx)
numerical_test_ds = apply_index(test_ds, node2idx, edge2idx)

## From `nerworkx` to `torch`

## Designing the Entity-Focused Graph

The goal is to convert the directed KG to the undirected one. Instead of passing the graph, the original triples can be used to bypass the unnecessary step of creating `networkx` graph.

In [ ]:
def create_entity_focused_graph(triples: Iterable[Dict[str, int]], num_entities: int) -> torch.Tensor:
    G_ef = nx.Graph() # undirected graph
    G_ef.add_nodes_from(range(num_entities))

    # Add edges for each triple (ignoring relation types)
    for triple in triples:
        h, r, t = triple.values()
        G_ef.add_edge(h, t)

    # Convert to edge_index
    edge_index_ef = torch.tensor(list(G_ef.edges)).t().contiguous()
    return edge_index_ef

## Designing the Relation-Focused Graph

Tong et al. $[1]$ proposed to build the relation-focused graph to learn potential dependence between adjacent relations in the KG.

To construct such a graph, first the _Relation Focused_ (RF) constraints are composed. RF constraints are triples $(r_s, e_p, r_o)$, where $r_s$ and $r_o$ are subjective and objective relations respectively both present in the relation list of the KG and connected by the same predicate entity $e_p$.

Then, out of all RF constraints only $\beta$ part is selected which has the largest $(r_s, r_o)$ mutual co-occurence in the graph.

In [ ]:
def extract_rf_constraints(triples: Iterable[Dict[str, int]]) -> Iterable[Dict[str, str]]:
    constraints = []

    # Build entity-centered lookup: entity -> [(in_relation, out_relation)]
    entity_context = {}

    for triple in triples:
        h, r, t = triple.values()
        # Add outgoing relations
        if h not in entity_context:
            entity_context[h] = {'in': [], 'out': []}
        entity_context[h]['out'].append((r, t))

        # Add incoming relations
        if t not in entity_context:
            entity_context[t] = {'in': [], 'out': []}
        entity_context[t]['in'].append((r, h))

    # Generate RF constraints: (in_relation, entity, out_relation)
    for entity, context in entity_context.items():
        for r_in, source in context['in']:
            for r_out, target in context['out']:
                constraints.append({
                    'head': r_in,
                    'relation': entity,
                    'tail': r_out
                })

    return constraints

In [ ]:
def calculate_cooccurrence_frequency(constraints: Iterable[Dict[str, int]]) -> Dict[Tuple[Any, Any], int]:
    cooccurrence = {}

    for constraint in constraints:
        r_s, e_p, r_o = constraint.values()
        pair = (r_s, r_o)  # This is directional

        if pair not in cooccurrence:
            cooccurrence[pair] = 0
        cooccurrence[pair] += 1

    return cooccurrence

In [ ]:
def create_relation_focused_graph(
    triples: Iterable[Dict[str, str]],
    num_entities: int,
    num_relations: int,
    beta=0.2
):

    # 1. Extract all RF constraints
    constraints = extract_rf_constraints(triples)

    # 2. Count co-occurrence frequencies (directional)
    cooccurrence = calculate_cooccurrence_frequency(constraints)

    # 3. Keep top beta% most frequent directional pairs
    sorted_pairs = sorted(cooccurrence.items(), key=lambda x: x[1], reverse=True)
    keep_count = int(len(sorted_pairs) * beta)
    kept_pairs = set([pair for pair, count in sorted_pairs[:keep_count]])

    # 4. Build graph with only kept constraints
    G_rf = nx.Graph()
    node_counter = 0
    entity_map, relation_map = {}, {}
    rf_to_original, rf_to_type = {}, {}

    for constraint in constraints:
        r_s, e_p, r_o = constraint.values()
        if (r_s, r_o) in kept_pairs:  # Check directional pair
            # Add nodes with unique IDs
            if e_p not in entity_map:
                entity_map[e_p] = node_counter
                G_rf.add_node(node_counter, type='entity', original_id=e_p)
                rf_to_original[node_counter] = ('entity', e_p)
                rf_to_type[node_counter] = 'entity'
                node_counter += 1

            if r_s not in relation_map:
                relation_map[r_s] = node_counter
                G_rf.add_node(node_counter, type='relation', original_id=r_s)
                rf_to_original[node_counter] = ('relation', r_s)
                rf_to_type[node_counter] = 'relation'
                node_counter += 1

            if r_o not in relation_map:
                relation_map[r_o] = node_counter
                G_rf.add_node(node_counter, type='relation', original_id=r_o)
                rf_to_original[node_counter] = ('relation', r_o)
                rf_to_type[node_counter] = 'relation'
                node_counter += 1

            # Add edges: r_s -- e_p -- r_o (undirected in graph structure)
            G_rf.add_edge(relation_map[r_s], entity_map[e_p])
            G_rf.add_edge(entity_map[e_p], relation_map[r_o])

    # Convert to tensors for batch processing
    rf_node_count = node_counter
    node_type_mask = torch.zeros(rf_node_count, dtype=torch.bool)
    for rf_id, node_type in rf_to_type.items():
        node_type_mask[rf_id] = (node_type == 'relation')

    # Convert to edge_index
    edge_index_rf = torch.tensor(list(G_rf.edges)).t().contiguous()

    return edge_index_rf, rf_to_original, rf_to_type, node_type_mask, entity_map, relation_map

## Combining all together

In [ ]:
def prepare_wge_graphs(
    train_triples: Iterable[Dict[str, int]],
    node2idx: Dict[str, int],
    edge2idx: Dict[int, str],
    beta=0.2
):

    # Build graphs using consistent indices
    edge_index_ef = create_entity_focused_graph(train_triples, len(node2idx))
    edge_index_rf, rf_to_original, rf_to_type, node_type_mask, entity_map, relation_map = create_relation_focused_graph(train_triples, len(node2idx), len(edge2idx), beta)

    return {
        'edge_index_ef': edge_index_ef, # [[head_idx_ef, ...], [tail_idx_ef, ...]]
        'edge_index_rf': edge_index_rf, # [[head_idx_rf, ...], [tail_idx_rf, ...]]
        'num_entities': len(node2idx),
        'num_relations': len(edge2idx),
        'node2idx': node2idx,
        'edge2idx': edge2idx,
        'rf_to_original': rf_to_original, # {node_idx_rf: (node type, node_idx) in KG}
        'node_type_mask': node_type_mask, # [True if node_idx_rf represents an entity not a relation]
        'rf_to_type': rf_to_type, # Additional to node type mask, {node_idx_rf: 'entity' or 'relation'}
        'entity_map': entity_map, # Additional to rf_to_original, {node_idx_rf: node_idx} if node_idx_rf is an entity
        'relation_map': relation_map # Additional to rf_to_original, {node_idx_rf: node_idx} if node_idx_rf is a relation
    }

May take some time

In [ ]:
result = prepare_wge_graphs(numerical_train_ds, node2idx, edge2idx)

## Embeddings

Since the KG does not consist of any feature vectors attached to nodes, the embeddings are learned. EF- and RF-focused graphs have distinct embeddings with the same dimensionality:

```
self.h_ef = nn.Embedding(
    num_embeddings=num_ef_nodes,
    embedding_dim=embedding_dim
)

# Usage
# self.h_ef(result['edge_index_ef'])

self.h_ef = nn.Embedding(
    num_embeddings=num_rf_nodes,
    embedding_dim=embedding_dim
)

# Usage
# self.h_rf(result['edge_index_rf'])
```

## Pipeline

**1. Graph Construction & Embedding Initialization**
- Build entity-focused (EF) and relation-focused (RF) graphs from original train triples (Sec 3.1)
- Initialize entity/relation embeddings via QGNN on both graphs (Eq 10-13)

**2. Supervised Training with Negative Sampling**  
- Augment training data with corrupted triples (head/tail replacement)
- Learn discriminative embeddings via backpropagation through:
  - Multi-layer quaternion scoring function (Eq 14-15)
  - Weighted binary cross-entropy loss (Eq 16)

**3. Evaluation via Ranking Metrics**
- For each test triple `(h,r,t)`, generate candidates:
  - Tail prediction: `(h,r,?)` with all entities as candidates
  - Head prediction: `(?,r,t)` with all entities as candidates  
  - Apply filtered setting (remove existing true triples)
- Compute ranking metrics:
  - **MRR**: Mean Reciprocal Rank of true triple
  - **Hits@K**: Proportion where true triple ranks ≤ K

# II. Preparation for training

Loads the YAGO3-10 dataset with pre-computed splits from HuggingFace using environment token authentication. Displays the triple count statistics for train, validation, and test sets.

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
from datasets import load_dataset
from huggingface_hub import login
from tqdm import tqdm

# Load HuggingFace token
load_dotenv(Path("env"))
hf_token = os.getenv("HF_TOKEN")

print("=" * 80)
print("WGE Data Loader - Using Pre-computed Splits")
print("=" * 80)

if hf_token:
    login(hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found - continuing without authentication")

# Load dataset with all splits
print("\nLoading YAGO3-10 dataset...")
ds = load_dataset("VLyb/YAGO3-10")

train_ds = ds['train']
val_ds = ds['validation']
test_ds = ds['test']

print(f"  Dataset loaded:")
print(f"  Train: {len(train_ds)} triples")
print(f"  Validation: {len(val_ds)} triples")
print(f"  Test: {len(test_ds)} triples")

WGE Data Loader - Using Pre-computed Splits
No HF_TOKEN found - continuing without authentication

Loading YAGO3-10 dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  Dataset loaded:
  Train: 1079040 triples
  Validation: 5000 triples
  Test: 5000 triples


## Build indexers

Creates entity and relation index mappings by processing all triples across train, validation, and test splits. Returns four dictionaries for bidirectional lookup between text labels and numerical indices.

In [ ]:
def build_indexers(train, val, test):
    all_triples = list(train) + list(val) + list(test)

    entities = set()
    relations = set()

    for triple in all_triples:
        h, r, t = triple.values()
        entities.add(h)
        entities.add(t)
        relations.add(r)

    entities = sorted(list(entities))
    relations = sorted(list(relations))

    node2idx = {node: i for i, node in enumerate(entities)}
    idx2node = {i: node for node, i in node2idx.items()}
    edge2idx = {edge: i for i, edge in enumerate(relations)}
    idx2edge = {i: edge for edge, i in edge2idx.items()}

    return node2idx, idx2node, edge2idx, idx2edge

node2idx, idx2node, edge2idx, idx2edge = build_indexers(train_ds, val_ds, test_ds)

print(f"  Indices created:")
print(f"  Entities: {len(node2idx)}")
print(f"  Relations: {len(edge2idx)}")

  Indices created:
  Entities: 123183
  Relations: 37


## Save indices

Saves the created entity and relation index mappings to four JSON files for persistence and later use in model training or evaluation.

In [ ]:
with open('node2idx.json', 'w') as f:
    json.dump(node2idx, f)
with open('idx2node.json', 'w') as f:
    json.dump(idx2node, f)
with open('edge2idx.json', 'w') as f:
    json.dump(edge2idx, f)
with open('idx2edge.json', 'w') as f:
    json.dump(idx2edge, f)


## Converting triples to numerical format

Converts the textual triples from each dataset split into numerical indices using the previously created entity and relation mappings. The function replaces entity and relation labels with their corresponding integer IDs for model consumption.

In [ ]:
def to_numerical(triples, node2idx, edge2idx):
    numerical = []
    for triple in triples:
        h, r, t = triple.values()
        numerical.append({
            'head': node2idx[h],
            'relation': edge2idx[r],
            'tail': node2idx[t]
        })
    return numerical

train_numerical = to_numerical(train_ds, node2idx, edge2idx)
val_numerical = to_numerical(val_ds, node2idx, edge2idx)
test_numerical = to_numerical(test_ds, node2idx, edge2idx)

print("Conversion complete")

Conversion complete


## Generating negative samples

This function generates adversarial training data by corrupting positive triples to create negative samples. For each positive triple, it randomly replaces either the head or tail entity with a different entity from the dataset, ensuring the corrupted entity isn't identical to the original. The process maintains a specified ratio of negative to positive samples (1:1 in this case), producing a balanced dataset with labeled samples where 1 indicates positive triples and 0 indicates negative ones. The final output is shuffled to prevent any ordering bias during model training.

In [ ]:
np.random.seed(42)

def generate_negative_samples(positive_triples, all_entity_ids, ratio=1):
    train_triples = []
    train_labels = []

    # Add positive samples
    train_triples.extend(positive_triples)
    train_labels.extend([1] * len(positive_triples))

    # Generate negative samples
    all_entity_ids = np.array(all_entity_ids)  # Convert to numpy array for faster sampling
    print(f"Generating {len(positive_triples) * ratio} negative samples...")

    for triple in tqdm(positive_triples, desc="Negative sampling", leave=False, dynamic_ncols=True, position=0):
        h, r, t = triple['head'], triple['relation'], triple['tail']

        for _ in range(ratio):
            if np.random.rand() > 0.5:
                # Corrupt head
                h_corrupt = np.random.choice(all_entity_ids)
                while h_corrupt == h:
                    h_corrupt = np.random.choice(all_entity_ids)
                train_triples.append({
                    'head': int(h_corrupt),
                    'relation': int(r),
                    'tail': int(t)
                })
            else:
                # Corrupt tail
                t_corrupt = np.random.choice(all_entity_ids)
                while t_corrupt == t:
                    t_corrupt = np.random.choice(all_entity_ids)
                train_triples.append({
                    'head': int(h),
                    'relation': int(r),
                    'tail': int(t_corrupt)
                })
            train_labels.append(0)

    # Shuffle
    indices = np.random.permutation(len(train_triples))
    train_triples = [train_triples[i] for i in indices]
    train_labels = [train_labels[i] for i in indices]

    return train_triples, train_labels

all_entity_ids = list(idx2node.keys())
train_triples, train_labels = generate_negative_samples(
    train_numerical,
    all_entity_ids,
    ratio=1
)

print(f"  Training data prepared:")
print(f"  Total samples: {len(train_triples)}")
print(f"  Positive: {sum(train_labels)}")
print(f"  Negative: {len(train_labels) - sum(train_labels)}")

Generating 1079040 negative samples...


  Training data prepared:
  Total samples: 2158080
  Positive: 1079040
  Negative: 1079040


## Saving training data

Saves the generated training triples and their corresponding labels to JSON files for persistent storage. This allows the processed dataset to be reused without needing to regenerate negative samples, facilitating efficient model training and experimentation.

In [ ]:
with open('train_triples.json', 'w') as f:
    json.dump(train_triples, f)
with open('train_labels.json', 'w') as f:
    json.dump(train_labels, f)
print("Training data saved")

Training data saved


## Create validation batches

Creates evaluation batches for link prediction by generating random negative candidates for both head and tail entities. The function samples from all possible entities, excluding true triples from the entire dataset to avoid false negatives. Each batch contains the test triple along with candidate lists for head and tail prediction, enabling efficient evaluation of model performance on the validation and test sets.

In [ ]:
import random
from tqdm import tqdm

def create_eval_batches(eval_triples, all_true_triples, all_entity_ids, max_candidates=1000):
    batches = []

    # Convert to set for faster membership testing
    all_true_triples_set = set(all_true_triples)

    for triple in tqdm(eval_triples, desc="Creating eval batches (random sampling)",
                       leave=False, dynamic_ncols=True, position=0):
        h, r, t = triple['head'], triple['relation'], triple['tail']

        # Sample random entities for tail prediction
        sampled_tails = random.sample(all_entity_ids, min(len(all_entity_ids), max_candidates * 2))
        tail_candidates = []
        for candidate_t in sampled_tails:
            if candidate_t != t and (h, r, candidate_t) not in all_true_triples_set:
                tail_candidates.append([int(h), int(r), int(candidate_t)])
                if len(tail_candidates) >= max_candidates:
                    break

        # Sample random entities for head prediction
        sampled_heads = random.sample(all_entity_ids, min(len(all_entity_ids), max_candidates * 2))
        head_candidates = []
        for candidate_h in sampled_heads:
            if candidate_h != h and (candidate_h, r, t) not in all_true_triples_set:
                head_candidates.append([int(candidate_h), int(r), int(t)])
                if len(head_candidates) >= max_candidates:
                    break

        batches.append({
            'test_triple': {'head': int(h), 'relation': int(r), 'tail': int(t)},
            'tail_candidates': tail_candidates,
            'head_candidates': head_candidates
        })

    return batches


print("Building true triples set for filtering...")
all_true_triples = set()
for triple in train_numerical + val_numerical + test_numerical:
    all_true_triples.add((triple['head'], triple['relation'], triple['tail']))
print(f"True triples: {len(all_true_triples)}")

# Create validation batches
print("\nCreating validation batches...")
val_batches = create_eval_batches(val_numerical, all_true_triples, all_entity_ids, 1000)
print(f"  Validation batches: {len(val_batches)}")

# Create test batches
print("\nCreating test batches...")
test_batches = create_eval_batches(test_numerical, all_true_triples, all_entity_ids, 5000)
print(f"  Test batches: {len(test_batches)}")

Building true triples set for filtering...
True triples: 1089040

Creating validation batches...


  Validation batches: 5000

Creating test batches...


  Test batches: 5000


## Saving evaluation batches

Saves the generated evaluation batches for validation and testing to JSON files, enabling efficient model evaluation by providing precomputed negative candidates for each test triple. These files can be loaded later during model evaluation without the need for real-time negative sampling.

In [ ]:
with open('validation_batches.json', 'w') as f:
    json.dump(val_batches, f)
with open('test_batches.json', 'w') as f:
    json.dump(test_batches, f)
print("  Evaluation batches saved")

  Evaluation batches saved


## Graph Construction for WGE Model

This cell constructs two complementary graph structures from the knowledge graph triples:

### 1. Entity-Focused Graph
Creates an undirected graph where entities are nodes and edges represent co-occurrence in triples, regardless of relation type. This graph captures entity connectivity patterns and supports neighborhood-based reasoning.

### 2. Relation-Focused Graph
Builds a bipartite graph that connects relations through shared entities (RF constraints). For each entity that appears with both an incoming relation (r_s) and outgoing relation (r_o), it creates patterns of the form r_s → entity → r_o. Only the top β% most frequent relation pairs are retained to focus on meaningful correlations while controlling graph density.

### Output
The function saves both graph structures along with their metadata to a pickle file, enabling efficient loading for model training. This dual-graph representation allows the WGE model to leverage both entity co-occurrence patterns and relation correlation patterns during embedding learning.


In [ ]:
import torch
import json
import numpy as np
from pathlib import Path
from collections import defaultdict
import networkx as nx
from typing import Dict, List, Tuple, Set
import pickle


def create_entity_focused_graph(triples: List[Dict], num_entities: int) -> torch.Tensor:
    print("Creating entity-focused graph...")

    edges = set()
    for triple in triples:
        h, t = triple['head'], triple['tail']
        # Add undirected edge
        edges.add((min(h, t), max(h, t)))

    if len(edges) == 0:
        print("Warning: No edges in entity-focused graph!")
        return torch.zeros((2, 0), dtype=torch.long)

    edge_list = list(edges)
    # Create bidirectional edges for undirected graph
    edge_index = torch.tensor([[e[0], e[1]] for e in edge_list] +
                              [[e[1], e[0]] for e in edge_list], dtype=torch.long).t()

    print(f"  Number of nodes: {num_entities}")
    print(f"  Number of edges: {edge_index.size(1)}")

    return edge_index


def extract_rf_constraints(triples: List[Dict]) -> List[Dict]:
    print("Extracting RF constraints...")

    # Build entity-centered lookup
    entity_context = defaultdict(lambda: {'in': [], 'out': []})

    for triple in triples:
        h, r, t = triple['head'], triple['relation'], triple['tail']
        # Outgoing: h --r--> t
        entity_context[h]['out'].append((r, t))
        # Incoming: h <--r-- t
        entity_context[t]['in'].append((r, h))

    # Generate RF constraints
    constraints = []
    for entity, context in entity_context.items():
        for r_in, source in context['in']:
            for r_out, target in context['out']:
                constraints.append({
                    'head': r_in,      # subjective relation
                    'relation': entity, # predicate entity
                    'tail': r_out      # objective relation
                })

    print(f"  Total RF constraints: {len(constraints)}")
    return constraints


def calculate_cooccurrence(constraints: List[Dict]) -> Dict[Tuple[int, int], int]:
    cooccurrence = defaultdict(int)

    for constraint in constraints:
        r_s, r_o = constraint['head'], constraint['tail']
        cooccurrence[(r_s, r_o)] += 1

    return dict(cooccurrence)


def create_relation_focused_graph(
    triples: List[Dict],
    num_entities: int,
    num_relations: int,
    beta: float = 0.2
) -> Tuple[torch.Tensor, Dict, Dict, torch.Tensor, Dict, Dict]:
    print("Creating relation-focused graph...")

    # Step 1: Extract RF constraints
    constraints = extract_rf_constraints(triples)

    # Step 2: Calculate co-occurrence
    cooccurrence = calculate_cooccurrence(constraints)

    # Step 3: Keep top beta% most frequent pairs
    sorted_pairs = sorted(cooccurrence.items(), key=lambda x: x[1], reverse=True)
    keep_count = max(1, int(len(sorted_pairs) * beta))
    kept_pairs = set([pair for pair, count in sorted_pairs[:keep_count]])

    print(f"  Keeping top {beta*100}% ({keep_count}/{len(sorted_pairs)}) relation pairs")

    # Step 4: Build graph with kept constraints
    node_counter = 0
    entity_map = {}
    relation_map = {}
    rf_to_original = {}
    rf_to_type = {}
    edges = set()

    for constraint in constraints:
        r_s, e_p, r_o = constraint['head'], constraint['relation'], constraint['tail']

        if (r_s, r_o) not in kept_pairs:
            continue

        # Add entity node
        if e_p not in entity_map:
            entity_map[e_p] = node_counter
            rf_to_original[node_counter] = ('entity', e_p)
            rf_to_type[node_counter] = 'entity'
            node_counter += 1

        # Add relation nodes
        if r_s not in relation_map:
            relation_map[r_s] = node_counter
            rf_to_original[node_counter] = ('relation', r_s)
            rf_to_type[node_counter] = 'relation'
            node_counter += 1

        if r_o not in relation_map:
            relation_map[r_o] = node_counter
            rf_to_original[node_counter] = ('relation', r_o)
            rf_to_type[node_counter] = 'relation'
            node_counter += 1

        # Add edges: r_s -- e_p -- r_o
        rf_e = entity_map[e_p]
        rf_rs = relation_map[r_s]
        rf_ro = relation_map[r_o]

        edges.add((min(rf_rs, rf_e), max(rf_rs, rf_e)))
        edges.add((min(rf_e, rf_ro), max(rf_e, rf_ro)))

    # Create node type mask
    num_rf_nodes = node_counter
    node_type_mask = torch.zeros(num_rf_nodes, dtype=torch.bool)
    for rf_id, node_type in rf_to_type.items():
        node_type_mask[rf_id] = (node_type == 'entity')

    # Create edge index
    if len(edges) == 0:
        print("Warning: No edges in relation-focused graph!")
        edge_index_rf = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_list = list(edges)
        # Bidirectional
        edge_index_rf = torch.tensor([[e[0], e[1]] for e in edge_list] +
                                     [[e[1], e[0]] for e in edge_list], dtype=torch.long).t()

    print(f"  Number of RF nodes: {num_rf_nodes}")
    print(f"    - Entity nodes: {node_type_mask.sum().item()}")
    print(f"    - Relation nodes: {(~node_type_mask).sum().item()}")
    print(f"  Number of RF edges: {edge_index_rf.size(1)}")

    return edge_index_rf, rf_to_original, rf_to_type, node_type_mask, entity_map, relation_map


def prepare_wge_graphs(data_dir: Path, beta: float = 0.2):
    print("=" * 80)
    print("Preparing WGE Graphs")
    print("=" * 80)

    # Load preprocessed data
    print("\nLoading preprocessed data...")

    with open(data_dir / 'node2idx.json', 'r') as f:
        node2idx = json.load(f)
    with open(data_dir / 'edge2idx.json', 'r') as f:
        edge2idx = json.load(f)
    with open(data_dir / 'train_triples.json', 'r') as f:
        train_triples = json.load(f)

    num_entities = len(node2idx)
    num_relations = len(edge2idx)

    print(f"  Number of entities: {num_entities}")
    print(f"  Number of relations: {num_relations}")
    print(f"  Number of training triples: {len(train_triples)}")

    # Create entity-focused graph
    print("\n" + "-" * 80)
    edge_index_ef = create_entity_focused_graph(train_triples, num_entities)

    # Create relation-focused graph
    print("\n" + "-" * 80)
    edge_index_rf, rf_to_original, rf_to_type, node_type_mask, entity_map, relation_map = \
        create_relation_focused_graph(train_triples, num_entities, num_relations, beta)

    # Save graphs
    print("\n" + "-" * 80)
    print("Saving graphs...")

    graph_data = {
        'edge_index_ef': edge_index_ef,
        'edge_index_rf': edge_index_rf,
        'rf_to_original': rf_to_original,
        'rf_to_type': rf_to_type,
        'node_type_mask': node_type_mask,
        'entity_map': entity_map,
        'relation_map': relation_map,
        'num_entities': num_entities,
        'num_relations': num_relations,
        'num_rf_nodes': edge_index_rf.max().item() + 1 if edge_index_rf.size(1) > 0 else 0
    }

    # Save as pickle for preserving types
    with open(data_dir / 'graph_data.pkl', 'wb') as f:
        pickle.dump(graph_data, f)

    print(f"  Saved to: {data_dir / 'graph_data.pkl'}")

    print("\n" + "=" * 80)
    print("Graph preparation complete!")
    print("=" * 80)

    return graph_data


def load_graph_data(data_dir: Path):
    with open(data_dir / 'graph_data.pkl', 'rb') as f:
        return pickle.load(f)


data_dir = Path(".")
prepare_wge_graphs(data_dir, beta=0.2)
print('Completed')


Preparing WGE Graphs

Loading preprocessed data...
  Number of entities: 123183
  Number of relations: 37
  Number of training triples: 2158080

--------------------------------------------------------------------------------
Creating entity-focused graph...
  Number of nodes: 123183
  Number of edges: 3669120

--------------------------------------------------------------------------------
Creating relation-focused graph...
Extracting RF constraints...


# III. Model

## Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
from typing import Dict, List, Tuple, Optional
import tqdm
from pathlib import Path


class Config:
    embedding_dim = 64
    num_layers = 1
    dropout = 0.0

    # Training
    learning_rate = 1e-2  # INCREASED - quaternion models need higher LR
    batch_size = 256  # REDUCED for GPU memory stability
    num_epochs = 10
    eval_every = 1  # Evaluate less frequently to save time
    warmup_steps = 100  # Warmup for stability

    # Graph construction
    beta = 0.2  # Percentage of kept RF constraints

    # Decoder weights
    alpha_0 = 0.6  # Weight for layer 0
    # alpha_k for k>0 computed as (1-alpha_0)/K

    # Data paths
    data_dir = Path(".")

    # Device - Force GPU usage
    device = torch.device('cuda')



## Quaternion Operations

This class implements core mathematical operations for quaternion-valued tensors used in the WGE model. Quaternions are represented as 4D vectors (real, i, j, k components) concatenated along the last dimension.

## Key Operations:

1. **Hamilton Product (`hamilton_product`)**: Performs non-commutative quaternion multiplication following the Hamilton formula: q ⊗ p = (qr·pr - qi·pi - qj·pj - qk·pk, ...)

2. **Element-wise Product (`element_wise_product`)**: Computes component-wise multiplication of two quaternions (qr*pr, qi*pi, qj*pj, qk*pk)

3. **Norm Calculation (`quaternion_norm`)**: Computes the Euclidean norm of the entire quaternion vector

4. **Normalization (`normalize_quaternion`)**: Normalizes quaternions to unit length by dividing by their norm

5. **Inner Product (`quaternion_inner_product`)**: Computes the dot product between two quaternions by summing products of corresponding components

These operations enable the model to perform hypercomplex algebra in the embedding space, capturing richer relational patterns than real-valued embeddings.

In [ ]:
class QuaternionOps:
    @staticmethod
    def hamilton_product(q, p):
        # Split into components
        dim = q.size(-1) // 4
        q_r, q_i, q_j, q_k = torch.split(q, dim, dim=-1)
        p_r, p_i, p_j, p_k = torch.split(p, dim, dim=-1)

        # Compute Hamilton product components
        r = q_r * p_r - q_i * p_i - q_j * p_j - q_k * p_k
        i = q_i * p_r + q_r * p_i - q_k * p_j + q_j * p_k
        j = q_j * p_r + q_k * p_i + q_r * p_j - q_i * p_k
        k = q_k * p_r - q_j * p_i + q_i * p_j + q_r * p_k

        return torch.cat([r, i, j, k], dim=-1)

    @staticmethod
    def element_wise_product(q, p):
        dim = q.size(-1) // 4
        q_r, q_i, q_j, q_k = torch.split(q, dim, dim=-1)
        p_r, p_i, p_j, p_k = torch.split(p, dim, dim=-1)

        # Element-wise multiplication of corresponding components
        r = q_r * p_r
        i = q_i * p_i
        j = q_j * p_j
        k = q_k * p_k

        return torch.cat([r, i, j, k], dim=-1)

    @staticmethod
    def quaternion_norm(q):
        # Compute sum of squares across all components
        return torch.sqrt(torch.sum(q**2, dim=-1, keepdim=True) + 1e-8)

    @staticmethod
    def normalize_quaternion(q):
        # Compute norm of entire quaternion vector
        norm = torch.sqrt(torch.sum(q**2, dim=-1, keepdim=True) + 1e-8)

        # Normalize the entire quaternion by its magnitude
        return q / norm

    @staticmethod
    def quaternion_inner_product(q, p):
        dim = q.size(-1) // 4
        q_r, q_i, q_j, q_k = torch.split(q, dim, dim=-1)
        p_r, p_i, p_j, p_k = torch.split(p, dim, dim=-1)

        return torch.sum(q_r * p_r + q_i * p_i + q_j * p_j + q_k * p_k, dim=-1)


## QGNN Layer

Implements a single layer of a Quaternion Graph Neural Network (QGNN) that operates on hypercomplex-valued node embeddings. The layer performs two main operations:

## 1. Quaternion Linear Transformation
Applies a Hamilton product-based transformation using four separate real-valued weight matrices (W_r, W_i, W_j, W_k) for each quaternion component. This preserves the algebraic structure of quaternion multiplication.

## 2. Graph Convolution
Performs neighborhood aggregation using normalized adjacency matrix (D⁻¹/²AD⁻¹/²) with optional edge weights. The aggregation sums normalized neighbor features followed by a hyperbolic tangent activation.

## Key Features:
- **Quaternion-aware**: Maintains the 4D structure of quaternion embeddings throughout computation
- **Normalization**: Applies degree-based normalization for stable gradient flow
- **Activation**: Uses tanh activation for non-linearity while preserving quaternion structure

This layer enables message passing on graphs while respecting the hypercomplex algebra of quaternion embeddings.

In [ ]:
class QGNNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super(QGNNLayer, self).__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim

        # Quaternion weight matrix W^(k),Q
        # Each component (r,i,j,k) has its own real-valued weight matrix
        self.W_r = nn.Linear(in_dim // 4, out_dim // 4, bias=False)
        self.W_i = nn.Linear(in_dim // 4, out_dim // 4, bias=False)
        self.W_j = nn.Linear(in_dim // 4, out_dim // 4, bias=False)
        self.W_k = nn.Linear(in_dim // 4, out_dim // 4, bias=False)

    def forward(self, x, edge_index, edge_weight=None):
        # Split quaternion into components
        dim = x.size(-1) // 4
        x_r, x_i, x_j, x_k = torch.split(x, dim, dim=-1)

        # Apply quaternion weight matrix (Hamilton product)
        h_r = self.W_r(x_r) - self.W_i(x_i) - self.W_j(x_j) - self.W_k(x_k)
        h_i = self.W_i(x_r) + self.W_r(x_i) - self.W_k(x_j) + self.W_j(x_k)
        h_j = self.W_j(x_r) + self.W_k(x_i) + self.W_r(x_j) - self.W_i(x_k)
        h_k = self.W_k(x_r) - self.W_j(x_i) + self.W_i(x_j) + self.W_r(x_k)

        h = torch.cat([h_r, h_i, h_j, h_k], dim=-1)

        # Aggregate neighbors (graph convolution)
        row, col = edge_index

        # Compute normalization: D^{-1/2} A D^{-1/2}
        deg = torch.zeros(x.size(0), device=x.device)
        deg.scatter_add_(0, row, torch.ones(row.size(0), device=x.device))
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0

        if edge_weight is None:
            edge_weight = torch.ones(edge_index.size(1), device=x.device)

        # Apply normalization to edge weights
        norm = deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]

        # Aggregate: sum of normalized neighbor features
        out = torch.zeros_like(h)
        out.index_add_(0, row, h[col] * norm.unsqueeze(-1))

        # Apply activation
        out = torch.tanh(out)

        return out



## WGE Encoder

Implements the dual-view encoder component of the WGE model that processes both entity-focused and relation-focused graphs simultaneously. The encoder maintains separate quaternion embeddings for each graph view and performs iterative message passing to capture complex relational patterns.

## Key Components:

### 1. **Embedding Layers**
- `h_ef`: Quaternion embeddings for all entities in the entity-focused graph
- `h_rf`: Quaternion embeddings for all nodes (both entities and relations) in the relation-focused graph  
- `h_relations`: Separate relation embeddings for easy extraction and use in downstream tasks

### 2. **Dual QGNN Architecture**
- **Entity-focused QGNN**: Processes the entity co-occurrence graph using neighborhood aggregation
- **Relation-focused QGNN**: Processes the relation correlation graph through shared entity nodes

### 3. **Cross-View Integration**
Performs element-wise product between entity embeddings from both views (Equation 11), enabling information exchange between the structural (EF) and semantic (RF) perspectives. The encoder follows Equations 10-13 from the original WGE paper to iteratively refine embeddings through K layers.

### 4. **Vectorized Operations**
Uses tensor-based mapping for efficient transfer of embeddings between the two graph views, avoiding slow Python loops and enabling GPU acceleration.

The encoder outputs K+1 layers of embeddings for both entities and relations, which will be combined in the decoder using learnable layer weights.

In [ ]:
class WGEEncoder(nn.Module):
    def __init__(self, config, num_entities, num_relations, num_rf_nodes, entity_map, relation_map):
        super(WGEEncoder, self).__init__()
        self.config = config
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.num_rf_nodes = num_rf_nodes
        self.num_layers = config.num_layers

        # Store mapping from original IDs to RF node IDs
        self.entity_map = entity_map  # {original_entity_id: rf_node_id}
        self.relation_map = relation_map  # {original_relation_id: rf_node_id}

        # Embeddings for entity-focused graph
        self.h_ef = nn.Embedding(num_entities, config.embedding_dim)

        # Embeddings for relation-focused graph (entities + relations as nodes)
        self.h_rf = nn.Embedding(num_rf_nodes, config.embedding_dim)

        # Separate embeddings for relations (for easy extraction)
        self.h_relations = nn.Embedding(num_relations, config.embedding_dim)

        # QGNN layers for entity-focused graph
        self.qgnn_ef_layers = nn.ModuleList([
            QGNNLayer(config.embedding_dim, config.embedding_dim)
            for _ in range(config.num_layers)
        ])

        # QGNN layers for relation-focused graph
        self.qgnn_rf_layers = nn.ModuleList([
            QGNNLayer(config.embedding_dim, config.embedding_dim)
            for _ in range(config.num_layers)
        ])

        self._init_embeddings()

    def _init_embeddings(self):
        nn.init.xavier_uniform_(self.h_ef.weight)
        nn.init.xavier_uniform_(self.h_rf.weight)
        nn.init.xavier_uniform_(self.h_relations.weight)

        # Normalize initial embeddings
        with torch.no_grad():
            self.h_ef.weight.data = QuaternionOps.normalize_quaternion(self.h_ef.weight.data)
            self.h_rf.weight.data = QuaternionOps.normalize_quaternion(self.h_rf.weight.data)
            self.h_relations.weight.data = QuaternionOps.normalize_quaternion(self.h_relations.weight.data)

    def forward(self, edge_index_ef, edge_index_rf, rf_to_original, node_type_mask):
        entity_embeds_per_layer = []
        relation_embeds_per_layer = []

        # Initialize layer 0 embeddings (before QGNN layers)
        h_ef_0 = self.h_ef.weight  # Entity embeddings (num_entities, dim)
        h_rf_0 = self.h_rf.weight  # RF graph node embeddings (num_rf_nodes, dim)

        # Store initial embeddings (layer 0)
        entity_embeds_per_layer.append(h_ef_0)

        # Extract initial relation embeddings from h_relations
        h_rel_0 = self.h_relations.weight  # (num_relations, dim)
        relation_embeds_per_layer.append(h_rel_0)

        # Pre-compute entity/relation indices for faster lookup
        # Create tensors for vectorized operations
        entity_ids_in_rf = torch.tensor(list(self.entity_map.keys()), dtype=torch.long, device=h_ef_0.device)
        rf_node_ids_for_entities = torch.tensor(list(self.entity_map.values()), dtype=torch.long, device=h_ef_0.device)

        relation_ids_in_rf = torch.tensor(list(self.relation_map.keys()), dtype=torch.long, device=h_ef_0.device)
        rf_node_ids_for_relations = torch.tensor(list(self.relation_map.values()), dtype=torch.long, device=h_ef_0.device)

        # Current states
        h_ef_k = h_ef_0
        h_rf_k = h_rf_0

        # Iterate through QGNN layers
        for k in range(self.num_layers):
            # === Update RF graph embeddings (Equation 12) ===
            h_rf_prime = self.qgnn_rf_layers[k](h_rf_k, edge_index_rf)

            # === Combine EF and RF views for entity nodes (Equation 11) ===
            # Initialize with ones (identity for entities not in RF)
            h_entity_from_rf = torch.ones_like(h_ef_k)  # (num_entities, dim)

            # Vectorized: Map RF entity embeddings back to all entities
            if len(entity_ids_in_rf) > 0:
                h_entity_from_rf[entity_ids_in_rf] = h_rf_prime[rf_node_ids_for_entities]

            # Element-wise product: h_u,ef^(k),Q * h'_u,rf^(k),Q
            h_ef_combined = QuaternionOps.element_wise_product(h_ef_k, h_entity_from_rf)

            # === Update EF graph embeddings (Equation 10) ===
            h_ef_k = self.qgnn_ef_layers[k](h_ef_combined, edge_index_ef)

            # Store updated embeddings
            entity_embeds_per_layer.append(h_ef_k)

            # Extract relation embeddings from updated RF graph (Equation 13)
            h_rel_k = self.h_relations.weight.clone()  # Start with base embeddings

            # Vectorized: Update relations that are in RF graph
            if len(relation_ids_in_rf) > 0:
                h_rel_k[relation_ids_in_rf] = h_rf_prime[rf_node_ids_for_relations]

            relation_embeds_per_layer.append(h_rel_k)

            # Update RF state for next iteration (Equation 13)
            h_rf_k = h_rf_prime.clone()
            if len(entity_ids_in_rf) > 0:
                h_rf_k[rf_node_ids_for_entities] = h_ef_k[entity_ids_in_rf]

        return entity_embeds_per_layer, relation_embeds_per_layer



## QuatE Decoder

Implements the quaternion-based knowledge graph completion decoder that computes plausibility scores for (head, relation, tail) triples. This decoder extends the QuatE scoring function to handle multi-layer embeddings from the WGE encoder.

## Core Operations:

### 1. **Layer-wise Scoring**
For each layer k (0 to K):
- Normalizes relation embeddings to unit quaternions (Equation 14)
- Computes Hamilton product between head embeddings and normalized relation embeddings
- Calculates inner product between the result and tail embeddings

### 2. **Weighted Aggregation** (Equation 15)
Combines scores from all layers using learnable weights:
- α₀: Weight for initial embeddings (layer 0)
- α₁ to αₖ: Equal weights for each QGNN layer, computed as (1 - α₀)/K

### 3. **Quaternion Scoring Function**
Uses hypercomplex algebra to capture rich relational patterns:
- Hamilton product models complex relation transformations
- Inner product measures compatibility between transformed head and tail embeddings

The decoder outputs a scalar score for each triple, where higher scores indicate more plausible relationships, enabling both training (with binary labels) and evaluation (ranking candidates).

In [ ]:
class QuatEDecoder(nn.Module):
    def __init__(self, config):
        super(QuatEDecoder, self).__init__()
        self.config = config

        # Compute layer weights (Equation 15)
        # alpha_0 for initial embeddings, remaining weight split across K QGNN layers
        # Total layers = 1 (initial) + K (QGNN layers) = K+1
        self.alphas = [config.alpha_0] + \
                      [(1.0 - config.alpha_0) / config.num_layers] * config.num_layers
        # Now we have K+1 weights: [alpha_0, alpha_1, ..., alpha_K]

    def forward(self, h_heads, h_relations, h_tails):
        scores = []

        for k in range(len(h_heads)):
            # Normalize relation (Equation 14: h_r^{a,(k)})
            h_r_norm = QuaternionOps.normalize_quaternion(h_relations[k])

            # Compute (h_h * h_r^a)
            h_hr = QuaternionOps.hamilton_product(h_heads[k], h_r_norm)

            # Compute (h_h * h_r^a) • h_t
            score_k = QuaternionOps.quaternion_inner_product(h_hr, h_tails[k])

            scores.append(self.alphas[k] * score_k)

        # Weighted sum (Equation 15)
        final_score = torch.stack(scores, dim=0).sum(dim=0)

        return final_score


## Complete WGE Model

Integrates the encoder and decoder components into a unified end-to-end model for knowledge graph completion. The model follows the complete WGE pipeline described in the original paper.

## Forward Pass Pipeline:

### 1. **Encoding Phase**
Passes the dual-graph structure through the WGE encoder to generate multi-layer embeddings:
- Entity embeddings per layer (K+1 layers)
- Relation embeddings per layer (K+1 layers)

### 2. **Batch Processing**
Extracts specific embeddings for the input batch of triples:
- `h_heads_per_layer`: Head entity embeddings across all layers
- `h_relations_per_layer`: Relation embeddings across all layers  
- `h_tails_per_layer`: Tail entity embeddings across all layers

### 3. **Decoding Phase**
Passes the extracted layer-wise embeddings through the QuatE decoder to compute final plausibility scores using weighted combination across layers.

## Model Characteristics:
- **End-to-end**: Processes from raw graph structure to final prediction scores
- **Dual-graph aware**: Leverages both entity-focused and relation-focused graph views
- **Multi-layer**: Captures different levels of relational patterns through layer aggregation
- **Hypercomplex**: Uses quaternion algebra for rich representation learning

The model outputs scalar scores for each input triple, suitable for both training (binary classification) and evaluation (ranking).

In [ ]:
class WGEModel(nn.Module):
    def __init__(self, config, num_entities, num_relations, num_rf_nodes, entity_map, relation_map):
        super(WGEModel, self).__init__()
        self.config = config

        self.encoder = WGEEncoder(config, num_entities, num_relations, num_rf_nodes, entity_map, relation_map)
        self.decoder = QuatEDecoder(config)

    def forward(self, triples, edge_index_ef, edge_index_rf, rf_to_original, node_type_mask):
        # Encode
        entity_embeds_per_layer, relation_embeds_per_layer = self.encoder(
            edge_index_ef, edge_index_rf, rf_to_original, node_type_mask
        )

        # Extract embeddings for batch
        batch_size = triples.size(0)
        h_heads_per_layer = []
        h_relations_per_layer = []
        h_tails_per_layer = []

        for k in range(len(entity_embeds_per_layer)):
            h_heads_per_layer.append(entity_embeds_per_layer[k][triples[:, 0]])
            h_relations_per_layer.append(relation_embeds_per_layer[k][triples[:, 1]])
            h_tails_per_layer.append(entity_embeds_per_layer[k][triples[:, 2]])

        # Decode
        scores = self.decoder(h_heads_per_layer, h_relations_per_layer, h_tails_per_layer)

        return scores


# IV. Training

This function evaluates the WGE model by scoring candidate entities for head and tail prediction in each test triple. It calculates standard metrics including Mean Reciprocal Rank (MRR) and Hits@k (for k=1, 3, 10) for both prediction directions, providing a comprehensive assessment of the model's knowledge graph completion performance.

In [ ]:
def train_epoch(model, train_triples, train_labels, edge_index_ef, edge_index_rf,
                rf_to_original, node_type_mask, optimizer, config, scheduler=None):
    """
    Train for one epoch

    Args:
        model: WGE model
        train_triples: Training triples tensor (num_triples, 3)
        train_labels: Training labels tensor (num_triples,)
        edge_index_ef: Entity-focused graph
        edge_index_rf: Relation-focused graph
        rf_to_original: RF mapping
        node_type_mask: RF node types
        optimizer: Optimizer
        config: Configuration
        scheduler: Optional learning rate scheduler

    Returns:
        Average loss for the epoch
    """
    model.train()
    total_loss = 0.0
    num_batches = 0

    # Shuffle data
    num_samples = len(train_triples)
    indices = torch.randperm(num_samples)

    # Progress bar for batches
    num_total_batches = (num_samples + config.batch_size - 1) // config.batch_size
    pbar = tqdm.tqdm(total=num_total_batches, desc="Training", leave=False, dynamic_ncols=True, position=0)

    for start_idx in range(0, num_samples, config.batch_size):
        end_idx = min(start_idx + config.batch_size, num_samples)
        batch_indices = indices[start_idx:end_idx]

        batch_triples = train_triples[batch_indices].to(config.device)
        batch_labels = train_labels[batch_indices].to(config.device).float()

        # Forward pass
        optimizer.zero_grad()
        scores = model(batch_triples, edge_index_ef, edge_index_rf,
                      rf_to_original, node_type_mask)

        # Debug: check score distribution (first batch only)
        if num_batches == 1:
            print(f"\nScore statistics (first batch):")
            print(f"  Min: {scores.min().item():.4f}")
            print(f"  Max: {scores.max().item():.4f}")
            print(f"  Mean: {scores.mean().item():.4f}")
            print(f"  Std: {scores.std().item():.4f}")

        # Compute loss (Equation 16: weighted binary cross-entropy)
        probs = torch.sigmoid(scores)
        loss = -(batch_labels * torch.log(probs + 1e-8) +
                (1 - batch_labels) * torch.log(1 - probs + 1e-8)).mean()

        # Backward pass
        loss.backward()

        # Check gradients (debug)
        if num_batches == 1:
            grad_norms = []
            for name, param in model.named_parameters():
                if param.grad is not None:
                    grad_norm = param.grad.norm().item()
                    grad_norms.append((name, grad_norm))

            print("\nGradient norms (first batch):")
            for name, norm in grad_norms[:5]:  # Show first 5
                print(f"  {name}: {norm:.6f}")

        # Gradient clipping для стабильности обучения
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Update learning rate
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        num_batches += 1

        # Update progress bar
        current_lr = optimizer.param_groups[0]['lr']
        pbar.update(1)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{current_lr:.2e}'})

        # Clear CUDA cache periodically to prevent memory fragmentation
        if num_batches % 50 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    pbar.close()
    return total_loss / num_batches



## Evaluation Functions

This function performs model evaluation on validation or test data by ranking candidate entities for both head and tail prediction tasks. It computes standard knowledge graph completion metrics including MRR (Mean Reciprocal Rank) and Hits@10 for each prediction direction, providing a comprehensive assessment of the model's link prediction performance.

In [ ]:
def evaluate(model, eval_batches, edge_index_ef, edge_index_rf,
            rf_to_original, node_type_mask, config):
    """
    Evaluate model on validation/test set

    Args:
        model: WGE model
        eval_batches: List of evaluation batches
        edge_index_ef: Entity-focused graph
        edge_index_rf: Relation-focused graph
        rf_to_original: RF mapping
        node_type_mask: RF node types
        config: Configuration

    Returns:
        Dictionary with MRR and Hits@10 metrics
    """
    model.eval()

    ranks_head = []
    ranks_tail = []

    # Limit evaluation to prevent memory issues
    max_eval_batches = min(len(eval_batches), 5000)  # Evaluate on subset

    with torch.no_grad():
        for idx, batch in enumerate(tqdm.tqdm(eval_batches[:max_eval_batches], desc="Evaluating", leave=False, dynamic_ncols=True, position=0)):
            # Clear cache every 100 batches
            if idx % 100 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
            test_triple = batch['test_triple']
            h, r, t = test_triple['head'], test_triple['relation'], test_triple['tail']

            # === Tail prediction: (h, r, ?) ===
            tail_candidates = batch['tail_candidates']
            if len(tail_candidates) > 0:
                # Create candidate triples
                candidate_triples_tail = torch.tensor(
                    [[h, r, t]] + tail_candidates,
                    device=config.device
                )

                # Score candidates
                scores_tail = model(candidate_triples_tail, edge_index_ef, edge_index_rf,
                                  rf_to_original, node_type_mask)

                # Rank true triple (first one)
                true_score = scores_tail[0]
                rank = (scores_tail >= true_score).sum().item()
                ranks_tail.append(rank)

            # === Head prediction: (?, r, t) ===
            head_candidates = batch['head_candidates']
            if len(head_candidates) > 0:
                # Create candidate triples
                candidate_triples_head = torch.tensor(
                    [[h, r, t]] + head_candidates,
                    device=config.device
                )

                # Score candidates
                scores_head = model(candidate_triples_head, edge_index_ef, edge_index_rf,
                                  rf_to_original, node_type_mask)

                # Rank true triple (first one)
                true_score = scores_head[0]
                rank = (scores_head >= true_score).sum().item()
                ranks_head.append(rank)

    # Compute metrics
    ranks_all = ranks_head + ranks_tail
    mrr = np.mean([1.0 / r for r in ranks_all])
    hits_at_10 = np.mean([1.0 if r <= 10 else 0.0 for r in ranks_all])

    mrr_head = np.mean([1.0 / r for r in ranks_head]) if ranks_head else 0.0
    mrr_tail = np.mean([1.0 / r for r in ranks_tail]) if ranks_tail else 0.0
    hits_at_10_head = np.mean([1.0 if r <= 10 else 0.0 for r in ranks_head]) if ranks_head else 0.0
    hits_at_10_tail = np.mean([1.0 if r <= 10 else 0.0 for r in ranks_tail]) if ranks_tail else 0.0

    return {
        'MRR': mrr,
        'Hits@10': hits_at_10,
        'MRR_head': mrr_head,
        'MRR_tail': mrr_tail,
        'Hits@10_head': hits_at_10_head,
        'Hits@10_tail': hits_at_10_tail
    }

## Data Loading Functions

This function loads preprocessed training and evaluation data from JSON files, including entity and relation mappings, training triples with labels, and validation/test batches. It converts the training data into PyTorch tensors for efficient model training and evaluation.

In [ ]:
def load_preprocessed_data(data_dir):
    data_dir = Path(data_dir)

    # Load indices
    with open(data_dir / 'node2idx.json', 'r') as f:
        node2idx = json.load(f)
    with open(data_dir / 'edge2idx.json', 'r') as f:
        edge2idx = json.load(f)

    # Load training data
    with open(data_dir / 'train_triples.json', 'r') as f:
        train_triples_list = json.load(f)
    with open(data_dir / 'train_labels.json', 'r') as f:
        train_labels = json.load(f)

    # Load evaluation batches
    with open(data_dir / 'validation_batches.json', 'r') as f:
        validation_batches = json.load(f)
    with open(data_dir / 'test_batches.json', 'r') as f:
        test_batches = json.load(f)

    # Convert to tensors
    train_triples = torch.tensor([
        [t['head'], t['relation'], t['tail']] for t in train_triples_list
    ])
    train_labels = torch.tensor(train_labels)

    return {
        'node2idx': node2idx,
        'edge2idx': edge2idx,
        'train_triples': train_triples,
        'train_labels': train_labels,
        'validation_batches': validation_batches,
        'test_batches': test_batches
    }



This function implements the main training and evaluation pipeline for the WGE model. It loads the configuration and preprocessed data, displays key parameters, and prepares the model structure for training, while noting the required graph data that needs to be generated separately.

In [ ]:
def main():
    """Main training and evaluation pipeline"""
    print("=" * 80)
    print("WGE: Two-View Graph Neural Networks for Knowledge Graph Completion")
    print("=" * 80)

    # Load configuration
    config = Config()
    print(f"\nConfiguration:")
    print(f"  Device: {config.device}")
    print(f"  Embedding dimension: {config.embedding_dim}")
    print(f"  Number of layers: {config.num_layers}")
    print(f"  Learning rate: {config.learning_rate}")
    print(f"  Batch size: {config.batch_size}")
    print(f"  Number of epochs: {config.num_epochs}")
    print(f"  Beta (RF constraint ratio): {config.beta}")

    # Load preprocessed data
    print("\nLoading preprocessed data...")
    data = load_preprocessed_data(config.data_dir)

    print(f"  Number of entities: {len(data['node2idx'])}")
    print(f"  Number of relations: {len(data['edge2idx'])}")
    print(f"  Number of training triples: {len(data['train_triples'])}")
    print(f"  Number of validation batches: {len(data['validation_batches'])}")
    print(f"  Number of test batches: {len(data['test_batches'])}")

    # Load graph structures (must be computed from existing code)
    print("\nNote: You need to run the graph construction code first to generate:")
    print("  - edge_index_ef (entity-focused graph)")
    print("  - edge_index_rf (relation-focused graph)")
    print("  - rf_to_original (RF node mapping)")
    print("  - node_type_mask (RF node types)")
    print("\nThese should be saved and loaded here.")
    print("\nFor now, the model structure is ready. Complete the data pipeline to train.")

    print("\n" + "=" * 80)
    print("Model implementation complete!")
    print("=" * 80)


main()

## Training itself

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
from pathlib import Path
import time


def load_all_data(data_dir):
    data_dir = Path(data_dir)

    print("Loading data...")

    # Load indices
    with open(data_dir / 'node2idx.json', 'r') as f:
        node2idx = json.load(f)
    with open(data_dir / 'edge2idx.json', 'r') as f:
        edge2idx = json.load(f)

    # Load training data
    with open(data_dir / 'train_triples.json', 'r') as f:
        train_triples_list = json.load(f)
    with open(data_dir / 'train_labels.json', 'r') as f:
        train_labels = json.load(f)

    # Load evaluation batches
    with open(data_dir / 'validation_batches.json', 'r') as f:
        validation_batches = json.load(f)
    with open(data_dir / 'test_batches.json', 'r') as f:
        test_batches = json.load(f)

    # Load graph structures
    graph_data = load_graph_data(data_dir)

    # Convert to tensors
    train_triples = torch.tensor([
        [t['head'], t['relation'], t['tail']] for t in train_triples_list
    ])
    train_labels = torch.tensor(train_labels)

    print(f"  Entities: {len(node2idx)}")
    print(f"  Relations: {len(edge2idx)}")
    print(f"  Training samples: {len(train_triples)}")
    print(f"  Validation batches: {len(validation_batches)}")
    print(f"  Test batches: {len(test_batches)}")
    print(f"  RF nodes: {graph_data['num_rf_nodes']}")

    return {
        'node2idx': node2idx,
        'edge2idx': edge2idx,
        'train_triples': train_triples,
        'train_labels': train_labels,
        'validation_batches': validation_batches,
        'test_batches': test_batches,
        'graph_data': graph_data
    }


def train_wge(config=None):
    """
    Complete training pipeline for WGE model

    Args:
        config: Configuration object (uses default if None)
    """
    if config is None:
        config = Config()

    print("=" * 80)
    print("WGE Training Pipeline")
    print("=" * 80)

    # Configuration
    print(f"\nConfiguration:")
    print(f"  Device: {config.device}")
    print(f"  Embedding dimension: {config.embedding_dim}")
    print(f"  Number of layers: {config.num_layers}")
    print(f"  Learning rate: {config.learning_rate}")
    print(f"  Batch size: {config.batch_size}")
    print(f"  Epochs: {config.num_epochs}")
    print(f"  Beta: {config.beta}")
    print(f"  Alpha_0: {config.alpha_0}")

    # Load data
    print("\n" + "-" * 80)
    data = load_all_data(config.data_dir)

    # Move graph data to device
    print("\n" + "-" * 80)
    print("Moving data to device...")
    edge_index_ef = data['graph_data']['edge_index_ef'].to(config.device)
    edge_index_rf = data['graph_data']['edge_index_rf'].to(config.device)
    node_type_mask = data['graph_data']['node_type_mask'].to(config.device)
    rf_to_original = data['graph_data']['rf_to_original']

    train_triples = data['train_triples']
    train_labels = data['train_labels']

    # Initialize model
    print("\n" + "-" * 80)
    print("Initializing model...")
    model = WGEModel(
        config,
        num_entities=data['graph_data']['num_entities'],
        num_relations=data['graph_data']['num_relations'],
        num_rf_nodes=data['graph_data']['num_rf_nodes'],
        entity_map=data['graph_data']['entity_map'],
        relation_map=data['graph_data']['relation_map']
    )
    model = model.to(config.device)

    # Count parameters
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total parameters: {num_params:,}")

    # Initialize optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

    # Learning rate scheduler with warmup
    def get_lr_multiplier(step):
        if step < config.warmup_steps:
            return step / config.warmup_steps
        return 1.0

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr_multiplier)

    training_history = {
        'epochs': [],
        'losses': [],
        'val_mrr': [],
        'val_hits10': []
    }

    # Try to load checkpoint if exists
    start_epoch = 1
    checkpoint_path = config.data_dir / 'checkpoint.pt'
    if checkpoint_path.exists():
        print(f"\nLoading checkpoint from {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_mrr = checkpoint.get('best_mrr', 0.0)
        best_epoch = checkpoint.get('best_epoch', 0)
        training_history = checkpoint.get('training_history', {'epochs': [], 'losses': [], 'val_mrr': [], 'val_hits10': []})
        print(f"  Resuming from epoch {start_epoch}")
        print(f"  Best MRR so far: {best_mrr:.4f} (epoch {best_epoch})")
    else:
        print("\nNo checkpoint found, starting from scratch...")

    # Training loop
    print("\n" + "=" * 80)
    print("Training")
    print("=" * 80)

    best_mrr = 0.0
    best_epoch = 0

    start_time = time.time()

    # Progress bar for epochs
    epoch_pbar = tqdm.tqdm(range(start_epoch, config.num_epochs + 1), desc="Epochs", dynamic_ncols=True, position=0)

    for epoch in epoch_pbar:
        epoch_start = time.time()

        # Train
        loss = train_epoch(
            model, train_triples, train_labels,
            edge_index_ef, edge_index_rf, rf_to_original, node_type_mask,
            optimizer, config, scheduler
        )

        epoch_time = time.time() - epoch_start

        # Update epoch progress bar
        epoch_pbar.set_postfix({
            'loss': f'{loss:.4f}',
            'time': f'{epoch_time:.1f}s',
            'best_mrr': f'{best_mrr:.4f}'
        })

        # Save checkpoint after each epoch (overwrite previous)
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_mrr': best_mrr,
            'best_epoch': best_epoch,
            'training_history': training_history,
            'config': config
        }
        torch.save(checkpoint, checkpoint_path)

        training_history['epochs'].append(epoch)
        training_history['losses'].append(loss)

        # Log progress
        if epoch % config.eval_every == 0 or epoch == 1:
            print(f"\n{'='*60}")
            print(f"Epoch {epoch}/{config.num_epochs}")
            print(f"  Time: {epoch_time:.2f}s")
            print(f"  Loss: {loss:.4f}")

            # Evaluate on validation set
            print("  Evaluating on validation set...")
            val_metrics = evaluate(
                model, data['validation_batches'],
                edge_index_ef, edge_index_rf, rf_to_original, node_type_mask,
                config
            )

            print(f"  Validation Metrics:")
            print(f"    MRR: {val_metrics['MRR']:.4f}")
            print(f"    Hits@10: {val_metrics['Hits@10']:.4f}")
            print(f"    MRR (head): {val_metrics['MRR_head']:.4f}")
            print(f"    MRR (tail): {val_metrics['MRR_tail']:.4f}")
            print(f"    Hits@10 (head): {val_metrics['Hits@10_head']:.4f}")
            print(f"    Hits@10 (tail): {val_metrics['Hits@10_tail']:.4f}")

            # Save training history
            training_history['val_mrr'].append(val_metrics['MRR'])
            training_history['val_hits10'].append(val_metrics['Hits@10'])

            # Save best model
            if val_metrics['MRR'] > best_mrr:
                best_mrr = val_metrics['MRR']
                best_epoch = epoch

                checkpoint = {
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_mrr': best_mrr,
                    'config': config
                }

                torch.save(checkpoint, config.data_dir / 'best_model.pt')
                print(f"   New best model saved! (MRR: {best_mrr:.4f})")
            print(f"{'='*60}\n")
        else:
            # Record None for validation metrics
            training_history['val_mrr'].append(None)
            training_history['val_hits10'].append(None)

    epoch_pbar.close()

    total_time = time.time() - start_time

    print("\n" + "=" * 80)
    print("Training Complete!")
    print("=" * 80)
    print(f"  Total time: {total_time/60:.2f} minutes")
    print(f"  Best epoch: {best_epoch}")
    print(f"  Best validation MRR: {best_mrr:.4f}")

    # Save training history
    with open(config.data_dir / 'training_history.json', 'w') as f:
        json.dump(training_history, f, indent=2)
    print(f"  Training history saved to: {config.data_dir / 'training_history.json'}")

    # Final evaluation on test set
    print("\n" + "=" * 80)
    print("Final Evaluation on Test Set")
    print("=" * 80)

    # Load best model
    checkpoint = torch.load(config.data_dir / 'best_model.pt', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])

    print("Evaluating on test set...")
    test_metrics = evaluate(
        model, data['test_batches'],
        edge_index_ef, edge_index_rf, rf_to_original, node_type_mask,
        config
    )

    print(f"\nTest Metrics:")
    print(f"  MRR: {test_metrics['MRR']:.4f}")
    print(f"  Hits@10: {test_metrics['Hits@10']:.4f}")
    print(f"  MRR (head): {test_metrics['MRR_head']:.4f}")
    print(f"  MRR (tail): {test_metrics['MRR_tail']:.4f}")
    print(f"  Hits@10 (head): {test_metrics['Hits@10_head']:.4f}")
    print(f"  Hits@10 (tail): {test_metrics['Hits@10_tail']:.4f}")

    # Save test results
    with open(config.data_dir / 'test_results.json', 'w') as f:
        json.dump(test_metrics, f, indent=2)

    print(f"\nTest results saved to: {config.data_dir / 'test_results.json'}")

    print("\n" + "=" * 80)

    return model, test_metrics


def main():
    # Create custom config if needed
    config = Config()

    # You can modify config here if needed
    # config.embedding_dim = 128
    # config.num_layers = 3
    config.learning_rate = 1e-3
    # config.batch_size = 1024
    # config.num_epochs = 500

    # Train model
    model, metrics = train_wge(config)

    print("\n Training pipeline completed successfully!")


main()

WGE Training Pipeline

Configuration:
  Device: cuda
  Embedding dimension: 64
  Number of layers: 1
  Learning rate: 0.001
  Batch size: 256
  Epochs: 10
  Beta: 0.2
  Alpha_0: 0.6

--------------------------------------------------------------------------------
Loading data...
  Entities: 123183
  Relations: 37
  Training samples: 2158080
  Validation batches: 5000
  Test batches: 5000
  RF nodes: 122133

--------------------------------------------------------------------------------
Moving data to device...

--------------------------------------------------------------------------------
Initializing model...
  Total parameters: 15,704,640

No checkpoint found, starting from scratch...

Training


Training:   0%|          | 1/8430 [00:01<2:39:09,  1.13s/it, loss=0.6931, lr=1.00e-05]


Score statistics (first batch):
  Min: -0.0596
  Max: 0.0468
  Mean: -0.0008
  Std: 0.0191


Training:   0%|          | 2/8430 [00:01<1:34:05,  1.49it/s, loss=0.6933, lr=2.00e-05]


Gradient norms (first batch):
  encoder.h_ef.weight: 0.006619
  encoder.h_rf.weight: 0.000001
  encoder.h_relations.weight: 0.004407
  encoder.qgnn_ef_layers.0.W_r.weight: 0.000001
  encoder.qgnn_ef_layers.0.W_i.weight: 0.000001


...


Epoch 10/10
  Time: 2716.56s
  Loss: 0.1027
  Evaluating on validation set...


  Validation Metrics:
    MRR: 0.3406
    Hits@10: 0.5446
    MRR (head): 0.1600
    MRR (tail): 0.5212
    Hits@10 (head): 0.3538
    Hits@10 (tail): 0.7354


Epochs: 100%|██████████| 10/10 [9:30:47<00:00, 3424.80s/it, loss=0.1027, time=2716.6s, best_mrr=0.3403]


   New best model saved! (MRR: 0.3406)


Training Complete!
  Total time: 570.80 minutes
  Best epoch: 10
  Best validation MRR: 0.3406
  Training history saved to: training_history.json

Final Evaluation on Test Set
Evaluating on test set...



Test Metrics:
  MRR: 0.1929
  Hits@10: 0.3463
  MRR (head): 0.0604
  MRR (tail): 0.3254
  Hits@10 (head): 0.1322
  Hits@10 (tail): 0.5604

Test results saved to: test_results.json


 Training pipeline completed successfully!


# V. Inference

In [ ]:
!curl -L "https://drive.usercontent.google.com/download?id=1fVAEBL8y3EJXpUondQ4_t02FuVvp6sfD&export=download&confirm=t" -o best_model.pt
!curl -L "https://drive.usercontent.google.com/download?id=1HIA36CJfOFDbsez5kXFWCqukdjMTxImU&export=download&confirm=t" -o idx2edge.json
!curl -L "https://drive.usercontent.google.com/download?id=1-18JwIEMYTMEwqSPv8ABtASAMxKLxIcD&export=download&confirm=t" -o edge2idx.json
!curl -L "https://drive.usercontent.google.com/download?id=1XxeaBwsOE16BZOpPdMZ-NVkCCrsMuT-6&export=download&confirm=t" -o node2idx.json
!curl -L "https://drive.usercontent.google.com/download?id=1mRSiVChvZjmuxM7m8DG4lm3l4WQ6WP_B&export=download&confirm=t" -o idx2node.json
!curl -L "https://drive.usercontent.google.com/download?id=16DjjKMYSoVUE8fAlkPZiNU_Wsnh4VFcu&export=download&confirm=t" -o graph_data.pkl

In [ ]:
import torch
import json
import argparse
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import numpy as np
import pickle
import sys


class WGEInference:

    def __init__(self, model_path: str = "best_model.pt", data_dir: str = "."):
        self.data_dir = Path(data_dir)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self._load_indices()
        self._load_graphs()

        self._load_model(model_path)

    def _load_indices(self):
        with open(self.data_dir / 'node2idx.json', 'r') as f:
            self.node2idx = json.load(f)
        with open(self.data_dir / 'idx2node.json', 'r') as f:
            self.idx2node = {int(k): v for k, v in json.load(f).items()}
        with open(self.data_dir / 'edge2idx.json', 'r') as f:
            self.edge2idx = json.load(f)
        with open(self.data_dir / 'idx2edge.json', 'r') as f:
            self.idx2edge = {int(k): v for k, v in json.load(f).items()}

        self.num_entities = len(self.node2idx)
        self.num_relations = len(self.edge2idx)

        print(f"  Entities: {self.num_entities}")
        print(f"  Relations: {self.num_relations}")

    def _load_graphs(self):
        """Загрузка структур графов"""
        import pickle
        with open(self.data_dir / 'graph_data.pkl', 'rb') as f:
            graph_data = pickle.load(f)

        self.edge_index_ef = graph_data['edge_index_ef'].to(self.device)
        self.edge_index_rf = graph_data['edge_index_rf'].to(self.device)
        self.rf_to_original = graph_data['rf_to_original']
        self.node_type_mask = graph_data['node_type_mask'].to(self.device)
        self.entity_map = graph_data['entity_map']
        self.relation_map = graph_data['relation_map']
        self.num_rf_nodes = graph_data['num_rf_nodes']

        print(f"  EF graph edges: {self.edge_index_ef.size(1)}")
        print(f"  RF graph edges: {self.edge_index_rf.size(1)}")
        print(f"  RF nodes: {self.num_rf_nodes}")

    def _load_model(self, model_path: str):
        import __main__
        __main__.Config = Config

        checkpoint = torch.load(model_path, map_location=self.device, weights_only=False)

        self.config = Config()
        self.config.device = self.device

        if 'config' in checkpoint and checkpoint['config'] is not None:
            old_config = checkpoint['config']
            if hasattr(old_config, 'embedding_dim'):
                self.config.embedding_dim = old_config.embedding_dim
            if hasattr(old_config, 'num_layers'):
                self.config.num_layers = old_config.num_layers
            if hasattr(old_config, 'alpha_0'):
                self.config.alpha_0 = old_config.alpha_0

        self.model = WGEModel(
            self.config,
            num_entities=self.num_entities,
            num_relations=self.num_relations,
            num_rf_nodes=self.num_rf_nodes,
            entity_map=self.entity_map,
            relation_map=self.relation_map
        )

        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model = self.model.to(self.device)
        self.model.eval()

        if 'epoch' in checkpoint:
            print(f"  Epoch: {checkpoint['epoch']}")
        if 'best_mrr' in checkpoint:
            print(f"  Best MRR: {checkpoint['best_mrr']:.4f}")

        num_params = sum(p.numel() for p in self.model.parameters())
        print(f"  Parameters: {num_params:,}")

    def entity_to_idx(self, entity: str) -> Optional[int]:
        return self.node2idx.get(entity)

    def relation_to_idx(self, relation: str) -> Optional[int]:
        return self.edge2idx.get(relation)

    def idx_to_entity(self, idx: int) -> str:
        return self.idx2node.get(idx, f"<unknown_{idx}>")

    def idx_to_relation(self, idx: int) -> str:
        return self.idx2edge.get(idx, f"<unknown_{idx}>")

    def score_triple(self, head: str, relation: str, tail: str) -> Optional[float]:
        h_idx = self.entity_to_idx(head)
        r_idx = self.relation_to_idx(relation)
        t_idx = self.entity_to_idx(tail)

        if h_idx is None:
            print(f"Error: Entity '{head}' not found in vocabulary")
            return None
        if r_idx is None:
            print(f"Error: Relation '{relation}' not found in vocabulary")
            return None
        if t_idx is None:
            print(f"Error: Entity '{tail}' not found in vocabulary")
            return None

        triple = torch.tensor([[h_idx, r_idx, t_idx]], device=self.device)

        with torch.no_grad():
            score = self.model(
                triple,
                self.edge_index_ef,
                self.edge_index_rf,
                self.rf_to_original,
                self.node_type_mask
            )

        return score.item()

    def predict_tail(self, head: str, relation: str, top_k: int = 10) -> List[Tuple[str, float]]:
        h_idx = self.entity_to_idx(head)
        r_idx = self.relation_to_idx(relation)

        if h_idx is None or r_idx is None:
            return []

        all_tails = list(range(self.num_entities))
        triples = torch.tensor([[h_idx, r_idx, t] for t in all_tails], device=self.device)

        batch_size = 1000
        all_scores = []

        with torch.no_grad():
            for i in range(0, len(triples), batch_size):
                batch = triples[i:i + batch_size]
                scores = self.model(
                    batch,
                    self.edge_index_ef,
                    self.edge_index_rf,
                    self.rf_to_original,
                    self.node_type_mask
                )
                all_scores.append(scores.cpu())

        all_scores = torch.cat(all_scores)

        top_scores, top_indices = torch.topk(all_scores, min(top_k, len(all_scores)))

        results = []
        for idx, score in zip(top_indices, top_scores):
            entity = self.idx_to_entity(idx.item())
            results.append((entity, score.item()))

        return results

    def predict_head(self, relation: str, tail: str, top_k: int = 10) -> List[Tuple[str, float]]:
        r_idx = self.relation_to_idx(relation)
        t_idx = self.entity_to_idx(tail)

        if r_idx is None or t_idx is None:
            return []

        all_heads = list(range(self.num_entities))
        triples = torch.tensor([[h, r_idx, t_idx] for h in all_heads], device=self.device)

        batch_size = 1000
        all_scores = []

        with torch.no_grad():
            for i in range(0, len(triples), batch_size):
                batch = triples[i:i + batch_size]
                scores = self.model(
                    batch,
                    self.edge_index_ef,
                    self.edge_index_rf,
                    self.rf_to_original,
                    self.node_type_mask
                )
                all_scores.append(scores.cpu())

        all_scores = torch.cat(all_scores)

        top_scores, top_indices = torch.topk(all_scores, min(top_k, len(all_scores)))

        results = []
        for idx, score in zip(top_indices, top_scores):
            entity = self.idx_to_entity(idx.item())
            results.append((entity, score.item()))

        return results

    def batch_score(self, triples: List[Tuple[str, str, str]]) -> List[Optional[float]]:
        results = []
        for head, relation, tail in triples:
            score = self.score_triple(head, relation, tail)
            results.append(score)
        return results

    def rank_triple_with_candidates(self, true_triple: Tuple[int, int, int],
                                   candidates: List[Tuple[int, int, int]],
                                   prediction_type: str = 'tail') -> int:
        all_triples = torch.tensor([true_triple] + candidates, device=self.device)

        with torch.no_grad():
            scores = self.model(
                all_triples,
                self.edge_index_ef,
                self.edge_index_rf,
                self.rf_to_original,
                self.node_type_mask
            )

        true_score = scores[0]

        rank = (scores >= true_score).sum().item()

        return rank

In [ ]:
inference = WGEInference(model_path="best_model.pt")

inference.predict_tail("Albert_Einstein", "wasBornIn", top_k=10)

In [ ]:
inference = WGEInference(model_path="best_model.pt")

inference.score_triple('Leonid_Musin', 'wasBornIn', 'Moscow'), inference.score_triple('Leonid_Musin', 'wasBornIn', 'Chelyabinsk')

# References

$[1]$ V. Tong, D. Nguyen, D. Phung, and D. Nguyen, ”Two-view Graph Neural Networks for Knowledge Graph Completion.” The Semantic Web.
ESWC 2023. Lecture Notes in Computer Science, vol. 13870. Springer,
Cham. DOI: 10.1007/978-3-031-33455-916, 2023.